# WEPAStacks Inbound Congestion Prediction bằng MLP

Notebook này triển khai đầy đủ pipeline trong `MLP_training_specification_vi.txt`: kiểm tra dữ liệu, tổng hợp theo 15 phút, mô phỏng queue có giới hạn, tạo target 60 phút, chia tập theo thời gian, huấn luyện MLP, chọn threshold trên validation, đánh giá test và lưu artifacts.

Chạy lần lượt tất cả cell từ trên xuống. Notebook hỗ trợ VS Code/Jupyter và Google Colab.

In [ ]:
# Kiểm tra dependencies. Colab sẽ tự cài các package còn thiếu.
import importlib.util
import subprocess
import sys

PACKAGE_IMPORTS = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'scikit-learn': 'sklearn',
    'tensorflow': 'tensorflow',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'joblib': 'joblib',
}
missing_packages = [pkg for pkg, module in PACKAGE_IMPORTS.items() if importlib.util.find_spec(module) is None]
IN_COLAB = 'google.colab' in sys.modules

if missing_packages and IN_COLAB:
    print('Đang cài package còn thiếu:', ', '.join(missing_packages))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
elif missing_packages:
    raise ImportError(
        'Thiếu package: ' + ', '.join(missing_packages) +
        '. Trong terminal của VS Code, chạy: ' +
        sys.executable + ' -m pip install ' + ' '.join(missing_packages)
    )
else:
    print('Tất cả dependencies đã sẵn sàng.')

In [ ]:
# Imports, cấu hình và reproducibility
import json
import os
import random
import shutil
from pathlib import Path

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print(f'Không thể bật deterministic ops hoàn toàn: {exc}')

INTERVAL_SECONDS = 900
PREDICTION_HORIZON = 4
PREDICTION_HORIZON_SECONDS = PREDICTION_HORIZON * INTERVAL_SECONDS
DOCKS = [1, 2, 3, 4]
BUFFER_CAPACITY = 20
CONGESTION_THRESHOLD = 0.80
SERVICE_CAPACITY_MARGIN = 0.10
TRAIN_END_DAY = 60
VALIDATION_END_DAY = 74
MAX_EPOCHS = 100
BATCH_SIZE = 256
LEARNING_RATE = 0.001
EARLY_STOPPING_PATIENCE = 8

NUMERIC_FEATURES = ['buffer_fill_ratio', 'arrivals_15m', 'mean_arrivals_60m']
DOCK_FEATURES = ['dock_1', 'dock_2', 'dock_3', 'dock_4']
FEATURE_NAMES = NUMERIC_FEATURES + DOCK_FEATURES

print('TensorFlow:', tf.__version__)
print('Seed:', SEED)

## 1. Tìm, đọc và làm sạch dữ liệu

Thứ tự tìm file: thư mục hiện tại, `assignment_2/`, sau đó yêu cầu upload nếu đang chạy trên Colab.

In [ ]:
FILE_NAME = 'wepastacks_inbound_clean.csv'
candidate_paths = [Path(FILE_NAME), Path('assignment_2') / FILE_NAME]
data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None and IN_COLAB:
    from google.colab import files
    print(f'Không tìm thấy {FILE_NAME}. Vui lòng upload file.')
    uploaded = files.upload()
    if FILE_NAME in uploaded:
        data_path = Path(FILE_NAME)

if data_path is None:
    raise FileNotFoundError(
        f'Không tìm thấy {FILE_NAME}. Hãy đặt file cạnh notebook hoặc trong thư mục assignment_2/.'
    )

data_path = data_path.resolve()
raw_df = pd.read_csv(data_path)
raw_df.columns = raw_df.columns.str.strip().str.lower()

required_columns = {'arrival_time', 'dock'}
missing_columns = required_columns - set(raw_df.columns)
if missing_columns:
    raise ValueError(f'Thiếu cột bắt buộc: {sorted(missing_columns)}')

if 'type' in raw_df.columns:
    raw_df = raw_df[raw_df['type'].astype(str).str.strip().str.lower().eq('delivery')].copy()

raw_df['arrival_time'] = pd.to_numeric(raw_df['arrival_time'], errors='coerce')
raw_df['dock'] = pd.to_numeric(raw_df['dock'], errors='coerce')
raw_df = raw_df.dropna(subset=['arrival_time', 'dock']).copy()
raw_df['arrival_time'] = raw_df['arrival_time'].astype(np.int64)
raw_df['dock'] = raw_df['dock'].astype(np.int64)
raw_df = raw_df[raw_df['arrival_time'].ge(0) & raw_df['dock'].isin(DOCKS)].copy()
raw_df['time_bin_15m'] = raw_df['arrival_time'] // INTERVAL_SECONDS
raw_df = raw_df.sort_values(['arrival_time', 'dock']).reset_index(drop=True)

assert raw_df[['arrival_time', 'dock']].notna().all().all()
assert set(raw_df['dock'].unique()).issubset(set(DOCKS))
assert (raw_df['arrival_time'] >= 0).all()

print('Đường dẫn:', data_path)
print(f'Số inbound pallet events hợp lệ: {len(raw_df):,}')
display(raw_df.head())
display(raw_df['dock'].value_counts().sort_index().rename('pallet_count').to_frame())

## 2. Gộp arrivals và bổ sung zero-arrival intervals

In [ ]:
arrivals = (
    raw_df.groupby(['time_bin_15m', 'dock'])
    .size()
    .rename('arrivals_15m')
)

all_bins = np.arange(
    raw_df['time_bin_15m'].min(),
    raw_df['time_bin_15m'].max() + 1,
    dtype=np.int64,
)
complete_index = pd.MultiIndex.from_product(
    [all_bins, DOCKS], names=['time_bin_15m', 'dock']
)
interval_df = arrivals.reindex(complete_index, fill_value=0).reset_index()
interval_df['arrivals_15m'] = interval_df['arrivals_15m'].astype(np.int64)

dock_count_per_bin = interval_df.groupby('time_bin_15m')['dock'].nunique()
if not dock_count_per_bin.eq(len(DOCKS)).all():
    raise AssertionError('Mỗi time_bin_15m phải có đúng bốn dock.')
if (interval_df['arrivals_15m'] < 0).any():
    raise AssertionError('arrivals_15m không được âm.')

interval_df['interval_start_seconds'] = interval_df['time_bin_15m'] * INTERVAL_SECONDS
interval_df['day'] = interval_df['interval_start_seconds'] // 86_400 + 1
interval_df['hour'] = (interval_df['interval_start_seconds'] % 86_400) // 3_600
interval_df = interval_df.sort_values(['dock', 'time_bin_15m']).reset_index(drop=True)

interval_df['mean_arrivals_60m'] = (
    interval_df.groupby('dock', sort=False)['arrivals_15m']
    .transform(lambda values: values.rolling(window=4, min_periods=4).mean())
)

print(f'Số intervals: {len(all_bins):,}; số dòng dock-interval: {len(interval_df):,}')
display(interval_df.head(8))

## 3. Ước tính service rate chỉ từ training period

Công thức: `max(1, ceil((1 + margin) × mean arrivals))`. Không dùng validation hoặc test.

In [ ]:
service_training = interval_df[interval_df['day'].le(TRAIN_END_DAY)]
mean_arrivals_by_dock = service_training.groupby('dock')['arrivals_15m'].mean().reindex(DOCKS)
service_rates = {
    int(dock): max(1, int(np.ceil((1.0 + SERVICE_CAPACITY_MARGIN) * mean_arrivals_by_dock.loc[dock])))
    for dock in DOCKS
}

service_rate_table = pd.DataFrame({
    'mean_arrivals_15m_training': mean_arrivals_by_dock,
    'service_rate': pd.Series(service_rates),
})
display(service_rate_table)

# Giá trị tham khảo của dataset hiện tại là {1: 25, 2: 2, 3: 1, 4: 1}.
if len(raw_df) == 205_258:
    assert service_rates == {1: 25, 2: 2, 3: 1, 4: 1}, 'Service rate khác kết quả tham khảo.'

## 4. Mô phỏng bounded queue và tạo target congestion

Queue được mô phỏng riêng từng dock, bắt đầu từ 0 và luôn bị giới hạn bởi capacity. Target chỉ nhìn bốn giá trị queue trong tương lai; các biến tương lai không được đưa vào feature.

In [ ]:
interval_df['queue_length'] = 0
interval_df['overflow_pallets'] = 0

for dock in DOCKS:
    dock_mask = interval_df['dock'].eq(dock)
    arrivals_for_dock = interval_df.loc[dock_mask, 'arrivals_15m'].to_numpy()
    queue = 0
    queue_values = []
    overflow_values = []

    for arrivals_now in arrivals_for_dock:
        remaining = max(0, queue + int(arrivals_now) - service_rates[dock])
        overflow = max(0, remaining - BUFFER_CAPACITY)
        queue = min(remaining, BUFFER_CAPACITY)
        queue_values.append(queue)
        overflow_values.append(overflow)

    interval_df.loc[dock_mask, 'queue_length'] = queue_values
    interval_df.loc[dock_mask, 'overflow_pallets'] = overflow_values

interval_df['queue_length'] = interval_df['queue_length'].astype(np.int64)
interval_df['overflow_pallets'] = interval_df['overflow_pallets'].astype(np.int64)
interval_df['buffer_fill_ratio'] = interval_df['queue_length'] / BUFFER_CAPACITY

future_queue_columns = [
    interval_df.groupby('dock', sort=False)['queue_length'].shift(-step).rename(f'queue_t_plus_{step}')
    for step in range(1, PREDICTION_HORIZON + 1)
]
future_queue = pd.concat(future_queue_columns, axis=1)
interval_df['future_max_queue'] = future_queue.max(axis=1, skipna=False)

model_df = interval_df.dropna(subset=['mean_arrivals_60m', 'future_max_queue']).copy()
congestion_level = BUFFER_CAPACITY * CONGESTION_THRESHOLD
model_df['congestion'] = model_df['future_max_queue'].ge(congestion_level).astype(np.int8)

assert interval_df['queue_length'].between(0, BUFFER_CAPACITY).all()
assert interval_df['buffer_fill_ratio'].between(0, 1).all()
assert interval_df['overflow_pallets'].ge(0).all()
assert model_df[['mean_arrivals_60m', 'future_max_queue']].notna().all().all()

print(f'Số mẫu có đủ rolling và future horizon: {len(model_df):,}')
print(f'Congestion rate toàn bộ: {model_df["congestion"].mean():.2%}')
display(model_df.head())

## 5. Feature table và chronological split có khoảng cách một giờ

In [ ]:
dock_dummies = pd.get_dummies(model_df['dock'], prefix='dock', dtype=np.int8)
dock_dummies = dock_dummies.reindex(columns=DOCK_FEATURES, fill_value=0)
model_df = pd.concat([model_df, dock_dummies], axis=1)

train_boundary = TRAIN_END_DAY * 86_400
validation_boundary = VALIDATION_END_DAY * 86_400
t = model_df['interval_start_seconds']
train_mask = t.add(PREDICTION_HORIZON_SECONDS).lt(train_boundary)
validation_mask = t.ge(train_boundary) & t.add(PREDICTION_HORIZON_SECONDS).lt(validation_boundary)
test_mask = t.ge(validation_boundary)

assert not (train_mask & validation_mask).any()
assert not (train_mask & test_mask).any()
assert not (validation_mask & test_mask).any()
assert (t[train_mask] + PREDICTION_HORIZON_SECONDS < train_boundary).all()
assert (t[validation_mask] + PREDICTION_HORIZON_SECONDS < validation_boundary).all()

X_train = model_df.loc[train_mask, FEATURE_NAMES].copy()
X_validation = model_df.loc[validation_mask, FEATURE_NAMES].copy()
X_test = model_df.loc[test_mask, FEATURE_NAMES].copy()
y_train = model_df.loc[train_mask, 'congestion'].astype(np.int8).copy()
y_validation = model_df.loc[validation_mask, 'congestion'].astype(np.int8).copy()
y_test = model_df.loc[test_mask, 'congestion'].astype(np.int8).copy()
test_rows = model_df.loc[test_mask].copy()

split_summary = pd.DataFrame({
    'observations': [len(y_train), len(y_validation), len(y_test)],
    'congestion_rate': [y_train.mean(), y_validation.mean(), y_test.mean()],
}, index=['train', 'validation', 'test'])
display(split_summary.style.format({'congestion_rate': '{:.2%}'}))

for split_name, labels in [('train', y_train), ('validation', y_validation), ('test', y_test)]:
    if labels.nunique() != 2:
        raise AssertionError(f'{split_name} không chứa đủ hai target classes.')

if len(raw_df) == 205_258:
    assert (len(y_train), len(y_validation), len(y_test)) == (22_628, 5_360, 5_532)

## 6. StandardScaler và balanced class weights

Scaler chỉ fit trên ba numeric features của training set. One-hot features được giữ nguyên. Class weights cũng chỉ được tính từ `y_train`.

In [ ]:
scaler = StandardScaler()
X_train.loc[:, NUMERIC_FEATURES] = scaler.fit_transform(X_train[NUMERIC_FEATURES])
X_validation.loc[:, NUMERIC_FEATURES] = scaler.transform(X_validation[NUMERIC_FEATURES])
X_test.loc[:, NUMERIC_FEATURES] = scaler.transform(X_test[NUMERIC_FEATURES])

X_train_array = X_train[FEATURE_NAMES].to_numpy(dtype=np.float32)
X_validation_array = X_validation[FEATURE_NAMES].to_numpy(dtype=np.float32)
X_test_array = X_test[FEATURE_NAMES].to_numpy(dtype=np.float32)
y_train_array = y_train.to_numpy(dtype=np.float32)
y_validation_array = y_validation.to_numpy(dtype=np.float32)
y_test_array = y_test.to_numpy(dtype=np.int8)

class_counts = y_train.value_counts().sort_index()
n_classes = len(class_counts)
class_weights = {
    int(label): len(y_train) / (n_classes * int(count))
    for label, count in class_counts.items()
}

assert FEATURE_NAMES == list(X_train.columns)
assert np.isfinite(X_train_array).all()
print('Feature order:', FEATURE_NAMES)
print('Balanced class weights:', class_weights)

## 7. Xây dựng và huấn luyện MLP

In [ ]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(FEATURE_NAMES),), name='features'),
    tf.keras.layers.Dense(32, activation='relu', name='hidden_1'),
    tf.keras.layers.Dropout(0.20, seed=SEED, name='dropout'),
    tf.keras.layers.Dense(16, activation='relu', name='hidden_2'),
    tf.keras.layers.Dense(1, activation='sigmoid', name='congestion_probability'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='binary_accuracy'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(curve='PR', name='pr_auc'),
    ],
)
model.summary()

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=EARLY_STOPPING_PATIENCE,
    restore_best_weights=True,
    verbose=1,
)

history = model.fit(
    X_train_array,
    y_train_array,
    validation_data=(X_validation_array, y_validation_array),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1,
)
history_df = pd.DataFrame(history.history)
history_df.index.name = 'epoch_index'
history_df.insert(0, 'epoch', np.arange(1, len(history_df) + 1))
display(history_df.tail())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_df['epoch'], history_df['loss'], label='Train loss')
axes[0].plot(history_df['epoch'], history_df['val_loss'], label='Validation loss')
axes[0].set(title='Binary cross-entropy loss', xlabel='Epoch', ylabel='Loss')
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(history_df['epoch'], history_df['pr_auc'], label='Train PR-AUC')
axes[1].plot(history_df['epoch'], history_df['val_pr_auc'], label='Validation PR-AUC')
axes[1].set(title='PR-AUC history', xlabel='Epoch', ylabel='PR-AUC')
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 8. Chọn prediction threshold chỉ bằng validation set

In [ ]:
validation_probabilities = model.predict(X_validation_array, batch_size=BATCH_SIZE, verbose=0).ravel()
validation_precision, validation_recall, validation_thresholds = precision_recall_curve(
    y_validation_array.astype(np.int8), validation_probabilities
)

if validation_thresholds.size == 0:
    raise RuntimeError('Không thể chọn threshold: validation không tạo ra threshold hợp lệ.')

precision_at_threshold = validation_precision[:-1]
recall_at_threshold = validation_recall[:-1]
denominator = precision_at_threshold + recall_at_threshold
validation_f1 = np.divide(
    2 * precision_at_threshold * recall_at_threshold,
    denominator,
    out=np.zeros_like(denominator),
    where=denominator > 0,
)
best_threshold_index = int(np.argmax(validation_f1))
selected_threshold = float(validation_thresholds[best_threshold_index])
best_validation_f1 = float(validation_f1[best_threshold_index])

print(f'Selected threshold: {selected_threshold:.6f}')
print(f'Best validation F1: {best_validation_f1:.4f}')
print(f'Validation precision tại threshold: {precision_at_threshold[best_threshold_index]:.4f}')
print(f'Validation recall tại threshold: {recall_at_threshold[best_threshold_index]:.4f}')

## 9. Đánh giá duy nhất một lần trên test set

In [ ]:
test_probabilities = model.predict(X_test_array, batch_size=BATCH_SIZE, verbose=0).ravel()
test_predictions = (test_probabilities >= selected_threshold).astype(np.int8)

test_precision_value, test_recall_value, test_f1_value, test_support = (
    precision_recall_fscore_support(
        y_test_array, test_predictions, labels=[1], average=None, zero_division=0
    )
)
test_pr_auc = average_precision_score(y_test_array, test_probabilities)
test_roc_auc = roc_auc_score(y_test_array, test_probabilities)

test_metrics = pd.DataFrame({
    'metric': ['Precision', 'Recall', 'F1-score', 'Support', 'PR-AUC', 'ROC-AUC'],
    'value': [
        float(test_precision_value[0]),
        float(test_recall_value[0]),
        float(test_f1_value[0]),
        int(test_support[0]),
        float(test_pr_auc),
        float(test_roc_auc),
    ],
})
display(test_metrics)
print(classification_report(y_test_array, test_predictions, digits=4, zero_division=0))

matrix = confusion_matrix(y_test_array, test_predictions, labels=[0, 1])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(
    matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
    xticklabels=['Predicted 0', 'Predicted 1'],
    yticklabels=['Actual 0', 'Actual 1'], ax=axes[0],
)
axes[0].set(title='Test confusion matrix', xlabel='Prediction', ylabel='Actual')

test_curve_precision, test_curve_recall, _ = precision_recall_curve(y_test_array, test_probabilities)
axes[1].plot(test_curve_recall, test_curve_precision, label=f'AP / PR-AUC = {test_pr_auc:.4f}')
axes[1].axhline(y_test_array.mean(), color='gray', linestyle='--', label='Positive-class baseline')
axes[1].set(title='Test precision-recall curve', xlabel='Recall', ylabel='Precision', xlim=(0, 1), ylim=(0, 1.02))
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 10. Lưu model artifacts và tạo ZIP

In [ ]:
output_dir = data_path.parent / 'model_outputs'
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / 'mlp_congestion.keras'
scaler_path = output_dir / 'feature_scaler.pkl'
configuration_path = output_dir / 'model_configuration.json'
history_path = output_dir / 'training_history.csv'
predictions_path = output_dir / 'test_predictions.csv'

model.save(model_path)
joblib.dump(scaler, scaler_path)
history_df.to_csv(history_path, index=False)

model_configuration = {
    'feature_names': FEATURE_NAMES,
    'numeric_feature_names': NUMERIC_FEATURES,
    'interval_duration_seconds': INTERVAL_SECONDS,
    'prediction_horizon_intervals': PREDICTION_HORIZON,
    'prediction_horizon_minutes': PREDICTION_HORIZON_SECONDS // 60,
    'buffer_capacity_per_dock': BUFFER_CAPACITY,
    'service_capacity_margin': SERVICE_CAPACITY_MARGIN,
    'service_rate_by_dock': {str(dock): int(rate) for dock, rate in service_rates.items()},
    'congestion_threshold': CONGESTION_THRESHOLD,
    'congestion_level_pallets': float(congestion_level),
    'selected_prediction_threshold': selected_threshold,
    'training_end_day': TRAIN_END_DAY,
    'validation_end_day': VALIDATION_END_DAY,
    'random_seed': SEED,
}
with configuration_path.open('w', encoding='utf-8') as file:
    json.dump(model_configuration, file, ensure_ascii=False, indent=2)

prediction_columns = [
    'time_bin_15m', 'day', 'hour', 'dock', 'arrivals_15m',
    'mean_arrivals_60m', 'queue_length', 'buffer_fill_ratio', 'congestion',
]
test_output = test_rows[prediction_columns].copy()
test_output['predicted_probability'] = test_probabilities
test_output['predicted_congestion'] = test_predictions
test_output.to_csv(predictions_path, index=False)

zip_base = data_path.parent / 'mlp_congestion_results'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=output_dir))

expected_files = {
    'mlp_congestion.keras',
    'feature_scaler.pkl',
    'model_configuration.json',
    'training_history.csv',
    'test_predictions.csv',
}
actual_files = {path.name for path in output_dir.iterdir() if path.is_file()}
assert expected_files.issubset(actual_files)
assert FEATURE_NAMES == model_configuration['feature_names']
assert len(test_output) == len(y_test)
assert zip_path.exists()

print('Đã lưu artifacts tại:', output_dir)
for path in sorted(output_dir.iterdir()):
    print(' -', path.name)
print('ZIP:', zip_path)

### Lưu ý về đánh giá

Không điều chỉnh model hoặc threshold dựa trên kết quả test. Nếu thay đổi pipeline sau khi xem test, cần dành một tập độc lập mới cho đánh giá cuối cùng.